# Statistical downscaling with machine-learning models

Runs k-NN, random forest and ridge regression for every station of a catalogue: finds the
climate-model cell of each station, calibrates on the observed period, validates, and applies the
models to the scenario period.

Edit the **Configuration** cell. If the station file is not found, the notebook builds a small
**synthetic demo dataset**. Demo data are random and only show how the workflow runs; the scores
say nothing about the skill of the methods.

## 1. Configuration

In [ ]:
from pathlib import Path

STATION_DATA = Path("../data/stations.csv")        # Date (dd/mm/yyyy) + one column per station ID
STATION_CATALOG = Path("../data/catalog.csv")      # ID, Latitud, Longitud
MODEL_HISTORIC = Path("../data/gcm/historical")    # folder with .nc files
MODEL_RCP = Path("../data/gcm/rcp45")              # folder with .nc files
OUTPUT_FOLDER = Path("../output")

CAL_DATE = "2005-12-31"   # last day of calibration
VAL_DATE = "2012-12-31"   # last day of validation (None to skip)
APP_DATE = "2100-12-31"   # last day of the application
METHODS = ("knn", "forest", "linear")   # "mlp" is also available

## 2. Inputs (or the synthetic demo)

In [ ]:
import tempfile
import warnings

import matplotlib.pyplot as plt
import pandas as pd

from dsmodel import demo, io, run_catalog

warnings.filterwarnings("ignore", category=UserWarning)

if STATION_DATA.exists() and STATION_CATALOG.exists():
    args = dict(station_data=STATION_DATA, station_catalog=STATION_CATALOG, model_historic=MODEL_HISTORIC,
                model_rcp=MODEL_RCP, cal_date=CAL_DATE, val_date=VAL_DATE, app_date=APP_DATE)
    DEMO = False
else:
    print("DEMO MODE: no station file found; using random synthetic data.")
    args = demo.make_demo_dataset(tempfile.mkdtemp(prefix="dsmodel_demo_"))
    OUTPUT_FOLDER = Path(tempfile.mkdtemp(prefix="dsmodel_out_"))
    DEMO = True

stations = io.read_station_table(args["station_data"])
catalog = io.read_catalog(args["station_catalog"])
print(f"{stations.shape[1]} stations, {stations.index[0].date()} to {stations.index[-1].date()}")
catalog.head()

## 3. Run

In [ ]:
result = run_catalog(output_folder=OUTPUT_FOLDER, methods=METHODS, **args)
print(result.model_id, result.variable, result.experiment_id)
print("skipped:", result.skipped or "none")
result.validation.round(3)

## 4. Look at one station

In [ ]:
station_id = catalog.index[0]
fig, ax = plt.subplots(figsize=(8, 3.5))
for method, frame in result.predictions.items():
    annual = frame[station_id].resample("YE").sum() if result.variable == "pr" else frame[station_id].resample("YE").mean()
    annual.plot(ax=ax, label=method)
ax.axvline(pd.Timestamp(args["cal_date"]), color="grey", linestyle="--", label="end of calibration")
ax.set_title(f"Station {station_id}: annual values"); ax.legend()
plt.show()

## 5. Output files

In [ ]:
for path in result.files[-6:]:
    print(path)